# Notebook 02 -- Anthracnose HTR + PPI weight calibration

**Plan 4 Task 17, Step 1.** Calibrates the PPI sub-model weights (`anth`, `pm`, `hop`) against synthetic / retrospective CROPSAP outbreak data via ROC-AUC grid search (`mangoguard.risk.calibration.calibrate_weights`). Outputs `artifacts/ppi_weights.yaml` for the production recommender + a ROC curve PNG for the CREST report Section 8.

**Run sequence:**
1. Generate / load a long-format `(block_id, date, anth_component, pm_component, hop_component, outbreak)` DataFrame.
2. Call `calibrate_weights(...)` -- coarse 0.1-step then fine 0.01-step.
3. Plot ROC for both default and calibrated weights.
4. Persist the calibrated weights for use by `compute_ppi`.

Replace the synthetic data block with the real 2018-2024 retrospective table before CREST submission.

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from sklearn.metrics import roc_auc_score, roc_curve

from mangoguard.risk.calibration import calibrate_weights
from mangoguard.risk.ppi import DEFAULT_WEIGHTS

REPO_ROOT = Path("..").resolve()
ARTIFACTS_DIR = REPO_ROOT / "artifacts"
FIGS_DIR = ARTIFACTS_DIR / "figs"
ARTIFACTS_DIR.mkdir(exist_ok=True)
FIGS_DIR.mkdir(exist_ok=True)
print("Default weights:", DEFAULT_WEIGHTS)

## 1. Synthesise (or load) the calibration set

Each row is one (block, day) with the three pre-computed component scores plus the observed CROPSAP outbreak label.

PLACEHOLDER: replace with `pd.read_csv('data/retrospective_2018_2024.csv')` once curated.

In [ ]:
rng = np.random.default_rng(42)
N_ROWS = 500
rows = []
for _ in range(N_ROWS):
    a = rng.uniform(0.0, 1.0)
    p = rng.uniform(0.0, 1.0)
    h = rng.uniform(0.0, 1.0)
    # Synthetic outbreak label: weighted-sum-driven probability
    score = 0.6 * a + 0.25 * p + 0.15 * h
    outbreak = int(rng.uniform() < score)
    rows.append({"anth": a, "pm": p, "hop": h, "outbreak": outbreak})
df = pd.DataFrame(rows)
df.head()

In [ ]:
# Calibrate via ROC-AUC grid search
calibrated = calibrate_weights(df, label_column="outbreak")
print("Calibrated weights:", calibrated)

## 2. Compare default vs calibrated AUC + plot ROC

In [ ]:
y_true = df["outbreak"].values
default_score = (
    DEFAULT_WEIGHTS["anth"] * df["anth"]
    + DEFAULT_WEIGHTS["pm"] * df["pm"]
    + DEFAULT_WEIGHTS["hop"] * df["hop"]
).values
calibrated_score = (
    calibrated["anth"] * df["anth"] + calibrated["pm"] * df["pm"] + calibrated["hop"] * df["hop"]
).values
default_auc = roc_auc_score(y_true, default_score)
calibrated_auc = roc_auc_score(y_true, calibrated_score)
print(f"Default AUC:    {default_auc:.3f}")
print(f"Calibrated AUC: {calibrated_auc:.3f}")

fig, ax = plt.subplots(figsize=(6, 5))
for score, label in [
    (default_score, f"Default (AUC={default_auc:.3f})"),
    (calibrated_score, f"Calibrated (AUC={calibrated_auc:.3f})"),
]:
    fpr, tpr, _ = roc_curve(y_true, score)
    ax.plot(fpr, tpr, label=label)
ax.plot([0, 1], [0, 1], "k--", alpha=0.3, label="Random")
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("PPI ROC -- default vs calibrated weights")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(FIGS_DIR / "ppi_roc_default_vs_calibrated.png", dpi=150)
plt.show()

## 3. Persist calibrated weights

In [ ]:
weights_path = ARTIFACTS_DIR / "ppi_weights.yaml"
with open(weights_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(
        {
            "calibrated_on": "synthetic_v1_PLACEHOLDER",
            "n_samples": int(len(df)),
            "default": DEFAULT_WEIGHTS,
            "calibrated": calibrated,
            "default_auc": float(default_auc),
            "calibrated_auc": float(calibrated_auc),
        },
        f,
        default_flow_style=False,
    )
print(f"Wrote {weights_path}")